<a href="https://colab.research.google.com/github/Prasun6736/Natural_Language_Processing/blob/main/Glove_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%matplotlib inline
from matplotlib import pyplot as plt
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict

In [11]:

 #1. Set random seeds

torch.manual_seed(42)
np.random.seed(42)

#2. corpus

corpus = [
    "I enjoy playing football",
    "I like reading books",
    "I love playing football with my friends",
    "I prefer reading books at home",
    "I enjoy playing sports",
    "I like studying",
    "I am passionate about football",
    "I enjoy reading"
]

def tokenize(sentence):
    return re.findall(r"\b\w+\b", sentence.lower())

sentences = [tokenize(sentence) for sentence in corpus]

print("Tokenized corpus:")
for sentence in sentences:
    print(sentence)

# ------------------------------------------------------------
# 3. Build vocabulary
# ------------------------------------------------------------

word_counts = Counter(
    word
    for sentence in sentences
    for word in sentence
)

vocab = sorted(word_counts.keys())

word_to_id = {
    word: i for i, word in enumerate(vocab)
}

id_to_word = {
    i: word for word, i in word_to_id.items()
}

vocab_size = len(vocab)

print("\nVocabulary size:", vocab_size)
print("Vocabulary:", vocab)

# ------------------------------------------------------------
# 4. Build the co-occurrence matrix
# ------------------------------------------------------------

window_size = 2

cooccurrence = defaultdict(float)

for sentence in sentences:

    for i, target_word in enumerate(sentence):

        target_id = word_to_id[target_word]

        start = max(0, i - window_size)
        end = min(len(sentence), i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            context_word = sentence[j]
            context_id = word_to_id[context_word]

            distance = abs(i - j)

            # Closer context words receive greater weight
            cooccurrence[(target_id, context_id)] += (
                1.0 / distance
            )

# Display co-occurrence matrix
X = np.zeros((vocab_size, vocab_size), dtype=np.float32)

for (i, j), count in cooccurrence.items():
    X[i, j] = count

print("\nCo-occurrence matrix shape:", X.shape)
print("\nCo-occurrence matrix:")
print(X)

print("\nWord order:")
print(vocab)

# ------------------------------------------------------------
# 5. Extract observed word-context pairs
# ------------------------------------------------------------

pairs = list(cooccurrence.keys())

target_ids = torch.tensor(
    [i for i, j in pairs],
    dtype=torch.long
)

context_ids = torch.tensor(
    [j for i, j in pairs],
    dtype=torch.long
)

counts = torch.tensor(
    [cooccurrence[pair] for pair in pairs],
    dtype=torch.float32
)

# Log co-occurrence counts
log_counts = torch.log(counts)

print("\nNumber of observed pairs:", len(pairs))

# ------------------------------------------------------------
# 6. Define the GloVe model
# ------------------------------------------------------------

class GloVe(nn.Module):

    def __init__(self, vocab_size, embedding_dim):

        super().__init__()

        # Target word embeddings
        self.target_embeddings = nn.Embedding(
            vocab_size, embedding_dim
        )

        # Context word embeddings
        self.context_embeddings = nn.Embedding(
            vocab_size, embedding_dim
        )

        # Target biases
        self.target_bias = nn.Embedding(
            vocab_size, 1
        )

        # Context biases
        self.context_bias = nn.Embedding(
            vocab_size, 1
        )

        # Initialize embeddings
        nn.init.uniform_(
            self.target_embeddings.weight,
            -0.5 / embedding_dim,
            0.5 / embedding_dim
        )

        nn.init.uniform_(
            self.context_embeddings.weight,
            -0.5 / embedding_dim,
            0.5 / embedding_dim
        )

        nn.init.zeros_(self.target_bias.weight)
        nn.init.zeros_(self.context_bias.weight)

    def forward(self, target_ids, context_ids):

        target_vectors = self.target_embeddings(target_ids)

        context_vectors = self.context_embeddings(context_ids)

        # Dot product between target and context vectors
        dot_product = (
            target_vectors * context_vectors
        ).sum(dim=1)

        # Bias terms
        bias_target = self.target_bias(
            target_ids
        ).squeeze(1)

        bias_context = self.context_bias(
            context_ids
        ).squeeze(1)

        # Predicted log co-occurrence count
        prediction = (
            dot_product
            + bias_target
            + bias_context
        )

        return prediction


# ------------------------------------------------------------
# 7. Initialize model and optimizer
# ------------------------------------------------------------

embedding_dim = 20
epochs = 300
learning_rate = 0.03

x_max = 10.0
alpha = 0.75

model = GloVe(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)

optimizer = optim.Adagrad(
    model.parameters(),
    lr=learning_rate
)

# ------------------------------------------------------------
# 8. Define the GloVe weighting function
# ------------------------------------------------------------

weights = torch.where(
    counts < x_max,
    (counts / x_max) ** alpha,
    torch.ones_like(counts)
)

# ------------------------------------------------------------
# 9. Train the model
# ------------------------------------------------------------

for epoch in range(epochs):

    model.train()

    # Predict log co-occurrence counts
    predictions = model(
        target_ids,
        context_ids
    )

    # Weighted squared error
    loss = (
        weights * (predictions - log_counts) ** 2
    ).mean()

    # Backpropagation
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch + 1:3d}/{epochs}, "
            f"Loss = {loss.item():.6f}"
        )

# ------------------------------------------------------------
# 10. Extract final word embeddings
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    # Standard educational choice:
    # add target and context embeddings
    embeddings = (
        model.target_embeddings.weight
        + model.context_embeddings.weight
    ).cpu().numpy()

print("\nEmbedding matrix shape:", embeddings.shape)

# ------------------------------------------------------------
# 11. Get the embedding of a word
# ------------------------------------------------------------

def get_embedding(word):

    if word not in word_to_id:
        raise ValueError(
            f"Word '{word}' is not in the vocabulary."
        )

    return embeddings[word_to_id[word]]


print("\nEmbedding of 'football':")
print(get_embedding("football"))

print("\nEmbedding of 'reading':")
print(get_embedding("reading"))


# ------------------------------------------------------------
# 12. Calculate cosine similarity
# ------------------------------------------------------------

def cosine_similarity(word1, word2):

    v1 = get_embedding(word1)
    v2 = get_embedding(word2)

    denominator = (
        np.linalg.norm(v1)
        * np.linalg.norm(v2)
        + 1e-10
    )

    return float(np.dot(v1, v2) / denominator)


print("\nCosine similarities:")

print(
    "football vs playing:",
    round(cosine_similarity("football", "playing"), 4)
)

print(
    "reading vs books:",
    round(cosine_similarity("reading", "books"), 4)
)

print(
    "enjoy vs studying:",
    round(cosine_similarity("enjoy", "studying"), 4)
)


# ------------------------------------------------------------
# 13. Find most similar words
# ------------------------------------------------------------

def most_similar(word, top_k=5):

    vector = get_embedding(word)

    norms = np.linalg.norm(embeddings, axis=1)

    similarities = (
        embeddings @ vector
        / (
            norms * np.linalg.norm(vector)
            + 1e-10
        )
    )

    # Exclude the query word itself
    similarities[word_to_id[word]] = -np.inf

    # Get the highest similarity scores
    indices = np.argsort(similarities)[::-1][:top_k]

    return [
        (
            id_to_word[i],
            round(float(similarities[i]), 4)
        )
        for i in indices
    ]


print("\nWords similar to 'football':")
print(most_similar("football"))

print("\nWords similar to 'reading':")
print(most_similar("reading"))

print("\nWords similar to 'enjoy':")
print(most_similar("enjoy"))

# ------------------------------------------------------------
# 14. Inspect a word-context prediction
# ------------------------------------------------------------

def predict_log_count(target_word, context_word):

    i = word_to_id[target_word]
    j = word_to_id[context_word]

    with torch.no_grad():

        prediction = model(
            torch.tensor([i]),
            torch.tensor([j])
        )

    return float(prediction.item())


target = "playing"
context = "football"

actual_count = X[
    word_to_id[target],
    word_to_id[context]
]

predicted_log_count = predict_log_count(
    target,
    context
)

print("\nPrediction inspection:")
print("Target word:", target)
print("Context word:", context)
print("Observed co-occurrence count:", actual_count)
print("Predicted log count:", round(predicted_log_count, 4))

print("\nGloVe training completed!")


Tokenized corpus:
['i', 'enjoy', 'playing', 'football']
['i', 'like', 'reading', 'books']
['i', 'love', 'playing', 'football', 'with', 'my', 'friends']
['i', 'prefer', 'reading', 'books', 'at', 'home']
['i', 'enjoy', 'playing', 'sports']
['i', 'like', 'studying']
['i', 'am', 'passionate', 'about', 'football']
['i', 'enjoy', 'reading']

Vocabulary size: 19
Vocabulary: ['about', 'am', 'at', 'books', 'enjoy', 'football', 'friends', 'home', 'i', 'like', 'love', 'my', 'passionate', 'playing', 'prefer', 'reading', 'sports', 'studying', 'with']

Co-occurrence matrix shape: (19, 19)

Co-occurrence matrix:
[[0.  0.5 0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.
  0. ]
 [0.5 0.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0.  1.  0.  0.  0.  0.  0.
  0. ]
 [0.  0.  0.  1.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.5 0.  0.
  0. ]
 [0.  0.  1.  0.  0.  0.  0.  0.5 0.  0.5 0.  0.  0.  0.  0.5 2.  0.  0.
  0. ]
 [0.  0.  0.  0.  0.  0.5 0.  0.  3.  0.  0.  0.  0.  2.  0.  1.  0.5 0.
  